# 电商平台用户行为分析与 RFM 价值分层体系搭建

本 Notebook 基于 80,000 条用户行为数据，完成用户画像分析、消费行为分析与 RFM 用户价值分层。


<h1>一、项目背景与目标</h1><h2>1.1 项目背景</h2><p>当前电商行业获客成本持续攀升，粗放式运营已无法支撑平台长期增长，<strong>精细化用户运营</strong>成为提升平台营收、降低用户流失的核心抓手。</p><p>本项目基于电商平台用户全维度行为数据，完成用户全景画像搭建、用户价值分层与行为规律挖掘，为平台精细化运营提供数据支撑与可落地的策略建议。</p><h2>1.2 项目核心目标</h2><ul><li><p>完成数据清洗与预处理，保障分析数据的准确性与有效性；</p></li><li><p>搭建平台用户全景画像，明确核心用户群体的人口属性、地域、兴趣与品类偏好特征；</p></li><li><p>基于 RFM 经典模型完成用户价值分层，定位高价值用户、流失风险用户与潜力增长用户；</p></li><li><p>挖掘影响用户总消费的核心指标，为平台 GMV 增长提供分析参考。</p></li></ul><h2>1.3 数据说明</h2><p>数据来源：公开电商平台用户行为与属性数据集，并基于原始样本扩展生成。</p><p>数据规模：80,000 条用户样本，15 列字段（含索引列），覆盖用户属性、行为特征、消费价值三大维度。</p><table style="min-width: 50px;"><tbody><tr><th><p>字段分类</p></th><th><p>核心字段</p></th></tr><tr><td><p>用户属性</p></td><td><p>用户 ID、性别、年龄、地域、收入、兴趣标签</p></td></tr><tr><td><p>行为特征</p></td><td><p>上次登录距今天数、网站停留时长、浏览页面数、是否订阅邮件</p></td></tr><tr><td><p>消费特征</p></td><td><p>购买频率、平均订单价值、总消费额、产品品类偏好</p></td></tr></tbody></table>

<h1>二、分析流程与数据预处理</h1><h2>2.1 整体分析流程</h2><p>本项目严格遵循数据分析全流程规范，执行逻辑为：</p><p>数据预处理→用户全景画像 EDA→用户行为深度分析→RFM 用户价值分层→消费行为影响因素分析→业务策略输出</p><h2>2.2 数据预处理</h2><p>为避免「垃圾进、垃圾出」，保障分析结论的严谨性，执行以下预处理操作：</p><ul><li><p>数据清洗：剔除 2 条完全重复的用户数据、1 条消费额为负的逻辑错误数据；非核心字段缺失值采用众数填充，避免样本量损失。</p></li><li><p>异常值处理：基于电商业务逻辑，保留 18-70 岁的核心运营用户，剔除无民事行为能力、行为特征与主流群体差异极大的离群样本，避免干扰分析结论。</p></li><li><p>特征工程：基于原始字段，构建 RFM 三大核心指标（R：最近一次登录距今天数、F：购买频率、M：总消费额），为后续用户价值分群做数据准备。</p></li><li><p>可视化代码优化：针对用户上次登录距今天数分布可视化的代码报错问题，放弃原有封装函数，采用「手动绘制直方图 + KDE 曲线缩放匹配」的方案，既保留了分布特征与平滑趋势，又完美匹配直方图的用户数量刻度，最终实现了合规、准确的可视化效果，核心代码如下：</p></li></ul><p></p>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import gaussian_kde
# 设置中文显示，避免图表乱码
plt.rcParams['font.sans-serif'] = ['SimHei']  #  Windows用这个
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # Mac用这个，注释掉上面一行，打开这行
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
from pathlib import Path

data_path = Path.cwd() / 'data' / 'user_personalized_features.csv'
if not data_path.exists():
    data_path = Path.cwd().parent / 'data' / 'user_personalized_features.csv'

df = pd.read_csv(data_path)


In [ ]:
print("数据基本信息：")
print(df.info())
print("\n数据描述性统计：")
print(df.describe())

In [ ]:
print("\n各字段缺失值数量：")
print(df.isnull().sum())

In [ ]:
print("\n重复数据行数：", df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
df = df[(df['Age'] >= 18) & (df['Age'] <= 70)]  # 只保留18-70岁的合理用户
df = df[df['Purchase_Frequency'] >= 0]  # 购买频率不能为负
df = df[df['Total_Spending'] >= 0]  # 消费额不能为负

In [ ]:
df = df.reset_index(drop=True)
print(f"\n清洗后数据量：{df.shape[0]}行，{df.shape[1]}列")

In [ ]:
# 1. 性别分布
plt.figure(figsize=(8, 5))
gender_count = df['Gender'].value_counts()
sns.barplot(x=gender_count.index, y=gender_count.values, palette='Blues_d')
plt.title('平台用户性别分布', fontsize=14)
plt.xlabel('性别', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
# 给柱子加数值标签
for i, v in enumerate(gender_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户性别分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 年龄分层分布（把年龄分组，更有业务意义）
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 70], labels=['18-25岁', '26-35岁', '36-45岁', '46-55岁', '56岁以上'])
plt.figure(figsize=(10, 5))
age_count = df['Age_Group'].value_counts().sort_index()
sns.barplot(x=age_count.index, y=age_count.values, palette='Purples_d')
plt.title('平台用户年龄分层分布', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(age_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户年龄分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. 用户地域分布
plt.figure(figsize=(8, 5))
location_count = df['Location'].value_counts()
sns.barplot(x=location_count.index, y=location_count.values, palette='Greens_d')
plt.title('平台用户地域分布', fontsize=14)
plt.xlabel('居住地', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
for i, v in enumerate(location_count.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=10)
plt.savefig('用户地域分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. 用户兴趣分布
plt.figure(figsize=(12, 6))
interest_count = df['Interests'].value_counts()
sns.barplot(x=interest_count.values, y=interest_count.index, palette='Oranges_d')
plt.title('平台用户兴趣分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('兴趣标签', fontsize=12)
plt.savefig('用户兴趣分布.png', dpi=300, bbox_inches='tight')
plt.show()

<h1>三、核心分析过程与业务洞察</h1><h2>3.1 平台用户全景画像分析</h2><p>通过多维度交叉分析，明确平台核心用户的群体特征，为精细化运营提供基础用户地图。</p><h3>3.1.1 用户人口属性分布</h3><ul><li><p><strong>性别分布</strong>：平台男性用户 526 人，占比 52.7%；女性用户 474 人，占比 47.3%，男女用户结构均衡，无明显性别倾斜。运营策略需兼顾两性需求，同时可针对占比略高的男性用户，适度倾斜对应品类的运营资源。</p></li><li><p><strong>年龄分层分布</strong>：平台核心用户集中在 36-55 岁，其中 36-45 岁用户 231 人、46-55 岁用户 223 人，合计占比超 45%，是平台的营收主力群体；18-25 岁年轻用户仅 152 人，占比最低，是平台用户增长的核心潜力缺口。</p></li><li><p><strong>年龄与消费能力关联</strong>：各年龄层用户平均总消费均稳定在 2500-2600 元，其中 56 岁以上用户平均消费最高（2603 元），46-55 岁次之（2596 元），说明平台用户消费能力无明显年龄壁垒，中老年用户具备极强的消费潜力。</p></li></ul><h3>3.1.2 用户地域与偏好分布</h3><ul><li><p><strong>地域分布</strong>：郊区用户 349 人，占比最高；城市用户 344 人，农村用户 307 人。下沉市场（郊区 + 农村）用户合计占比超 65%，是平台的用户基本盘，运营需重点匹配下沉市场用户的消费习惯。</p></li><li><p><strong>兴趣与品类偏好</strong>：</p><ul><li><p>用户兴趣 TOP3：Sports（运动）、Fashion（时尚）、Food（美食）；</p></li><li><p>产品品类偏好 TOP3：Apparel（服饰）、Electronics（数码产品）、Books（图书）；</p></li><li><p>核心关联洞察：用户兴趣与品类偏好高度匹配，运动、时尚兴趣直接对应服饰品类的核心消费，为场景化推荐、品类联动运营提供了数据支撑。</p></li></ul></li></ul><p></p>

In [ ]:
# 1. 上次登录天数分布（判断用户活跃/流失情况）
# 提取数据并去除空值
data = df['Last_Login_Days_Ago'].dropna()
plt.figure(figsize=(10, 5))
# 绘制直方图，和原需求效果完全一致
n, bins, patches = plt.hist(data, bins=10, color='#1f77b4', edgecolor='white')
# 绘制KDE平滑曲线，匹配histplot的kde=True效果
kde = gaussian_kde(data)
x_range = np.linspace(bins.min(), bins.max(), 1000)
# 缩放KDE曲线，匹配直方图的用户数量刻度
plt.plot(x_range, kde(x_range) * len(data) * np.diff(bins)[0], color='#1f77b4', linewidth=2)

plt.title('用户上次登录距今天数分布', fontsize=14)
plt.xlabel('距上次登录天数', fontsize=12)
plt.ylabel('用户数量', fontsize=12)
plt.show()
# 2. 订阅用户vs非订阅用户的活跃度对比
plt.figure(figsize=(10, 5))
sns.boxplot(x='Newsletter_Subscription', y='Time_Spent_on_Site_Minutes', data=df, palette='Set2')
plt.title('邮件订阅用户与非订阅用户的网站停留时长对比', fontsize=14)
plt.xlabel('是否订阅邮件', fontsize=12)
plt.ylabel('网站停留时长（分钟）', fontsize=12)
plt.savefig('订阅用户活跃度对比.png', dpi=300, bbox_inches='tight')
plt.show()

<h2>3.2 用户行为深度分析</h2><h3>3.2.1 用户活跃度与流失特征分析</h3><p>从用户上次登录距今天数分布可得出核心洞察：</p><ol><li><p>平台用户登录天数集中在 15-25 天区间，峰值出现在 20 天左右，15 天内登录的活跃用户占比约 60%，平台整体用户活跃度处于行业中等水平；</p></li><li><p>超过 30 天未登录的用户占比约 18%，已进入流失状态，另有 10% 的流失预警用户（15-30 天未登录），平台近 1/4 的用户存在流失风险，用户留存与召回工作迫在眉睫。</p></li></ol><h3>3.2.2 用户粘性影响因素分析</h3><p>通过箱线图对比邮件订阅用户与非订阅用户的网站停留时长，得出核心结论：</p><ul><li><p>邮件订阅用户的网站停留时长中位数约 300 分钟，非订阅用户约 280 分钟，订阅用户的停留时长上限、四分位值均显著高于非订阅用户；</p></li><li><p>业务洞察：邮件订阅能有效提升用户活跃度与平台粘性，是低成本撬动用户留存的核心抓手，需重点提升全平台用户的邮件订阅率。</p></li></ul><p></p>

In [ ]:
# 1. 产品品类偏好分布
plt.figure(figsize=(12, 6))
category_count = df['Product_Category_Preference'].value_counts()
sns.barplot(x=category_count.values, y=category_count.index, palette='RdBu_d')
plt.title('用户产品品类偏好分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('产品品类', fontsize=12)
plt.savefig('品类偏好分布.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 不同年龄层的总消费均值对比
plt.figure(figsize=(10, 5))
age_spending = df.groupby('Age_Group')['Total_Spending'].mean().sort_index()
sns.barplot(x=age_spending.index, y=age_spending.values, palette='coolwarm')
plt.title('不同年龄层用户平均总消费', fontsize=14)
plt.xlabel('年龄分组', fontsize=12)
plt.ylabel('平均总消费（元）', fontsize=12)
for i, v in enumerate(age_spending.values):
    plt.text(i, v+50, f'{int(v)}元', ha='center', fontsize=10)
plt.savefig('年龄层消费对比.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1. 提取RFM三个核心指标
rfm_df = df[['User_ID', 'Last_Login_Days_Ago', 'Purchase_Frequency', 'Total_Spending']]
rfm_df.columns = ['User_ID', 'R', 'F', 'M']

# 2. 给RFM指标打分（1-5分，5分为最优）
# R值：越小越好，所以倒序打分
rfm_df['R_Score'] = pd.qcut(rfm_df['R'].rank(method='first'), q=5, labels=[5,4,3,2,1])
# F值：越大越好，正序打分
rfm_df['F_Score'] = pd.qcut(rfm_df['F'].rank(method='first'), q=5, labels=[1,2,3,4,5])
# M值：越大越好，正序打分
rfm_df['M_Score'] = pd.qcut(rfm_df['M'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# 3. 拼接RFM得分，生成用户标签
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

# 4. 用户价值分层（经典8分法，简化版，容易理解和讲解）
def user_level(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >=4 and f >=4 and m >=4:
        return '高价值用户'
    elif r >=4 and f >=4 and m <4:
        return '潜力用户'
    elif r >=4 and f <4 and m >=4:
        return '深耕用户'
    elif r >=4 and f <4 and m <4:
        return '新用户'
    elif r <4 and f >=4 and m >=4:
        return '流失预警用户'
    elif r <4 and f >=4 and m <4:
        return '需召回用户'
    elif r <4 and f <4 and m >=4:
        return '流失高价值用户'
    else:
        return '低价值用户'

rfm_df['User_Level'] = rfm_df.apply(user_level, axis=1)

# 5. 查看各用户群体数量分布
level_count = rfm_df['User_Level'].value_counts()
print("\n用户价值分层结果：")
print(level_count)

# 6. 可视化用户分层结果
plt.figure(figsize=(12, 6))
sns.barplot(x=level_count.values, y=level_count.index, palette='Spectral')
plt.title('RFM用户价值分层分布', fontsize=14)
plt.xlabel('用户数量', fontsize=12)
plt.ylabel('用户分层', fontsize=12)
for i, v in enumerate(level_count.values):
    plt.text(v+5, i, str(v), va='center', fontsize=10)
plt.savefig('RFM用户分层.png', dpi=300, bbox_inches='tight')
plt.show()

# 7. 把分层结果合并回原数据，方便后续分析
df = pd.merge(df, rfm_df[['User_ID', 'User_Level']], on='User_ID', how='left')

<h2>3.3 RFM 用户价值分层分析</h2><p>基于 RFM 模型完成用户价值分层。R 表示最近一次登录距今天数，数值越小越好；F 表示购买频率，数值越大越好；M 表示总消费额，数值越大越好。三个指标均采用等频分箱打分，并通过 <code>rank(method="first")</code> 处理重复边界值。</p><table style="min-width: 100px;"><tbody><tr><th><p>用户分层</p></th><th><p>用户数量</p></th><th><p>业务定位</p></th></tr><tr><td><p>低价值用户</p></td><td><p>16,696</p></td><td><p>低活跃、低复购、低消费力，控制运营投入</p></td></tr><tr><td><p>新用户</p></td><td><p>12,529</p></td><td><p>高活跃、低复购、低消费力，重点完成首单与复购转化</p></td></tr><tr><td><p>流失高价值用户</p></td><td><p>11,694</p></td><td><p>低活跃、低复购、高消费力，重点召回</p></td></tr><tr><td><p>需召回用户</p></td><td><p>11,422</p></td><td><p>低活跃、高复购、低消费力，批量触达召回</p></td></tr><tr><td><p>流失预警用户</p></td><td><p>8,188</p></td><td><p>低活跃、高复购、高消费力，高风险群体需及时唤醒</p></td></tr><tr><td><p>潜力用户</p></td><td><p>7,353</p></td><td><p>高活跃、高复购、低消费力，重点提升客单价</p></td></tr><tr><td><p>深耕用户</p></td><td><p>7,081</p></td><td><p>高活跃、低复购、高消费力，重点提升复购率</p></td></tr><tr><td><p>高价值用户</p></td><td><p>5,037</p></td><td><p>近期活跃、高频复购、高消费力，平台核心资产</p></td></tr></tbody></table>

In [ ]:
# 筛选数值型字段，计算和总消费的相关性
num_cols = ['Age', 'Income', 'Last_Login_Days_Ago', 'Purchase_Frequency', 
            'Average_Order_Value', 'Time_Spent_on_Site_Minutes', 'Pages_Viewed', 'Total_Spending']
corr_df = df[num_cols].corr()

# 可视化相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('用户指标相关性热力图', fontsize=14)
plt.savefig('相关性热力图.png', dpi=300, bbox_inches='tight')
plt.show()

# 输出和总消费相关性最高的指标
print("\n和用户总消费相关性TOP5指标：")
print(corr_df['Total_Spending'].sort_values(ascending=False).head(6))

<h2>3.4 用户消费行为影响因素分析</h2><p>通过皮尔逊相关系数分析，观察各数值型指标与用户总消费之间的线性关系。</p><ul><li><p><strong>核心结论</strong>：在当前 80,000 条数据中，除总消费自身外，平均订单价值与总消费的相关性最高，相关系数约为 0.099。</p></li><li><p><strong>次要影响因素</strong>：上次登录距今天数、购买频率与总消费呈弱相关，可作为用户活跃与复购行为的辅助观察指标。</p></li><li><p><strong>补充洞察</strong>：年龄、收入、网站停留时长等指标与总消费的线性相关性较弱，后续可结合分群、交叉分析或建模方法进一步挖掘非线性关系。</p></li></ul>